# Analysis - what fine-tuning did to the odor tokens

Issue #8 asks what the embeddings of the odor tokens look like before step 2 and after it. Before, the model has
never read one; after, each has been read as a request thousands of times, through six adapters that started at
exactly zero.

The numbers come from `python -m scripts.analysis.embeddings`, which reads the committed `stage1` and `stage2`
checkpoints; nothing here trains.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = json.loads((ROOT / "output/analysis/embeddings.json").read_text())
cosine = np.array(data["cosine"])
moved = np.array(data["relative_change"])
overlap = np.array(data["neighbour_overlap"])
frequency = np.array(data["frequency"])
tokens = data["tokens"]
print(len(tokens), "odor tokens")


## 1. The embeddings barely moved

Cosine says how far the direction turned, and the relative move says how far the vector went in its own units. Both
are small.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(moved, bins=40, color="#4472C4")
ax.axvline(np.median(moved), color="#C00000", linestyle="--", linewidth=1, label=f"median {np.median(moved):.1%}")
ax.set_xlabel("relative move of the embedding")
ax.set_ylabel("tokens")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f"cosine: median {np.median(cosine):.4f}, minimum {cosine.min():.4f}")
print(f"relative move: median {np.median(moved):.1%}, mean {moved.mean():.1%}, most {moved.max():.1%}")
print(f"almost unchanged (cosine above 0.99): {np.mean(cosine > 0.99):.0%}")


## 2. The neighbours did move

A vector can stay close to itself and still trade places with its neighbours, because the space is crowded. For
each token, the share of its ten nearest neighbours that survive fine-tuning.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(overlap, bins=20, color="#4472C4")
ax.axvline(overlap.mean(), color="#C00000", linestyle="--", linewidth=1, label=f"mean {overlap.mean():.0%}")
ax.set_xlabel("share of the ten nearest neighbours kept")
ax.set_ylabel("tokens")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f"mean {overlap.mean():.0%}, median {np.median(overlap):.0%}, none kept {np.mean(overlap == 0):.0%}")


## 3. The map, before and after

A two-dimensional PCA of the 890 embeddings, coloured by how many molecules carry each token. The two panels share
the colour scale, not the axes.

In [ ]:
before = np.array(data["pca_before"])
after = np.array(data["pca_after"])
colour = np.log10(frequency + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=False, sharey=False)
for ax, points, title in zip(axes, (before, after), ("after pretraining", "after fine-tuning"), strict=True):
    shown = ax.scatter(points[:, 0], points[:, 1], c=colour, cmap="viridis", s=10)
    for token in np.argsort(-frequency)[:8]:
        ax.annotate(tokens[token], points[token], fontsize=7, alpha=0.9)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(shown, ax=axes, label="log10 molecules carrying the token", shrink=0.8)
plt.show()


## 4. Does closeness follow the data?

The question the map can only illustrate. For two odor tokens that label the same molecules, are their embeddings
closer? Cosine against the Jaccard overlap of their training molecules, over the 304 tokens with at least ten
molecules. The correlation is computed by `scripts/analysis/odor_map.py`.

In [ ]:
mapping = json.loads((ROOT / "output/analysis/odor_map.json").read_text())
print("tokens with at least", mapping["minimum_molecules"], "training molecules:", mapping["tokens"])
print("pairs:", mapping["pairs"], "| no molecule in common:", mapping["zero_overlap_pairs"])
print("cosine against label overlap, before step 2:", round(mapping["before"], 3))
print("cosine against label overlap, after step 2: ", round(mapping["after"], 3))
print("among the", mapping["shared_overlap_pairs"], "pairs that share a molecule:", round(mapping["spearman_shared"], 3))
bins = mapping["overlap_bins"]
fig, ax = plt.subplots(figsize=(7, 4))
values = [found["mean_cosine"] for found in bins]
bars = ax.bar(range(len(bins)), values, color="#1F3864", width=0.65)
ax.bar_label(bars, fmt="%.4f", fontsize=8, padding=2)
ax.set_xticks(range(len(bins)), [f"{found['label']}\n{found['pairs']:,} pairs" for found in bins], fontsize=8)
ax.set_ylim(0.94, max(values) + 0.004)
ax.set_xlabel("Jaccard overlap of the molecules the two tokens label")
ax.set_ylabel("mean cosine between the embeddings")
plt.tight_layout()
plt.show()

## 5. t-SNE, for the eye only

PCA keeps every token in one blob because the variance is spread over 256 dimensions. t-SNE spreads the local
neighbourhoods instead. It is a picture, not evidence: the correlation above is what the claim rests on.

In [ ]:
before = np.array(mapping["tsne_before"])
after = np.array(mapping["tsne_after"])
names = mapping["top_tokens"][:15]
positions = mapping["top_positions"][:15]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, points, title in zip(axes, (before, after), ("before step 2", "after step 2"), strict=True):
    ax.scatter(points[:, 0], points[:, 1], s=12, color="#B4C7E7")
    for name, position in zip(names, positions, strict=True):
        ax.annotate(name, points[position], fontsize=8, color="#1F3864")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 6. Where the request is carried

The six request adapters, one per layer, start at exactly zero and are never given weight decay. If the embeddings
hardly moved, this is where the conditioning must live.

In [ ]:
norms = data["adapter_norm"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, len(norms["stage2"]) + 1), norms["stage2"], color="#1F3864")
ax.set_xlabel("layer")
ax.set_ylabel("L2 norm of the adapter")
ax.set_title("the adapters after fine-tuning (they start at zero)")
plt.tight_layout()
plt.show()
print("stage1:", [round(v, 1) for v in norms["stage1"]])
print("stage2:", [round(v, 1) for v in norms["stage2"]])


## Appendix: the heatmap

Forty tokens against each other, on a scale that starts at 0.90. The field has no blocks; that is the same finding
as the bar chart, drawn 780 times.

In [ ]:
heatmap = np.array(mapping["heatmap"])
names = mapping["heatmap_tokens"]
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(heatmap, cmap="RdBu_r", vmin=0.90, vmax=1.0)
ax.set_xticks(range(len(names)), names, rotation=90, fontsize=7)
ax.set_yticks(range(len(names)), names, fontsize=7)
figure.colorbar(image, ax=ax, shrink=0.8, label="cosine")
ax.set_title("Cosine between the forty most frequent odor tokens, scale starts at 0.90")
plt.tight_layout()
plt.show()

## Reading.

The odor token embeddings come out of fine-tuning almost exactly where they went in: a median relative move of 3.1
percent, a cosine of 0.999, and not one token below 0.99. The two PCA maps are therefore indistinguishable by eye,
which is the finding, not a failure of the figure.

The neighbours tell a subtler story. Only 68 percent of each token's ten nearest neighbours survive, so while the
vectors stayed put, the local order changed in a third of the cases. In a space of 890 tokens that are mostly
near-synonyms, a three percent move is enough to reshuffle who sits next to whom.

The map does not follow the label data. Two thirds of the 46,056 pairs, 30,710 of them, label no molecule in
common, and the raw Spearman of -0.103 is mostly that tie mass. Among the 15,346 pairs that do share a molecule it
is +0.162, and the mean cosine moves only from 0.9470 to 0.9509 as the overlap goes from almost none to above 0.1.
Frequent tokens sit slightly further apart, which is why sharing a molecule at all comes with a slightly lower
cosine. As a description of meaning, the map says nothing.

What did move is the six adapters. They start at exactly zero, by construction, and end at norms of 4.3, 4.8, 5.6,
6.4, 7.5 and 8.9 from the first layer to the sixth. The monotone rise is what layer decay predicts: with 0.8 per
layer, the first layer learns at a third of the rate of the last, so the same training leaves it a smaller adapter.
The request is therefore carried by what is added to the state, not by a new meaning given to the odor words.

## Verdict

Fine-tuning did not relearn the odor vocabulary; it learned how to use it. That is the same conclusion the ablation
in the earlier notes reached from the other direction, where adding the adapters dropped the loss by 0.114 nats and
the embeddings alone had hardly helped.
